In [10]:
import os
import sys
import json
import torch

src_root = os.path.abspath("../..")   # from src/examples/milad -> src
if src_root not in sys.path:
    sys.path.insert(0, src_root)

print("src_root =", src_root)
print("datasets exists:", os.path.exists(os.path.join(src_root, "datasets")))
print("models exists:", os.path.exists(os.path.join(src_root, "models")))

from datasets.image_text_dataset import POPE_train_Dataset, POPE_test_Dataset
from models import get_model_class
from helpers.utils import set_seed

# -----------------------------
# CONFIG
# -----------------------------
model_name_or_path = "llava-hf/llava-1.5-7b-hf"  # or "Qwen/Qwen2-VL-7B-Instruct"
processor_name = model_name_or_path

data_dir = "/research/hal-afsharim/learn-to-steer/data/pope/train"
annotation_file = "annotations.json"
dataset_name = "pope_train"                       # or "pope_test"
split = "all"                                     # train: all/adversarial/popular/random
dataset_size = -1
seed = 0

max_new_tokens = 100   # POPE extraction scripts use 100
output_json = "pope_contrastive_with_continuations.json"



def join_prefix_and_cont(prefix, cont):
    prefix = prefix.strip()
    cont = cont.strip()
    if not cont:
        return prefix
    # add a space unless continuation begins with punctuation
    if cont[0] in ",.!?;:":
        return f"{prefix}{cont}"
    return f"{prefix} {cont}"

# -----------------------------
# DATASET
# -----------------------------
if dataset_name == "pope_train":
    ds = POPE_train_Dataset(
        data_dir=data_dir,
        annotation_file=annotation_file,
        split=split,
        dataset_size=dataset_size,
        seed=seed,
        dataset_name=dataset_name,
        mode="train",
        prompt_template="llava",
    )
    ann_path = os.path.join(data_dir, annotation_file)
    with open(ann_path, "r") as f:
        ann_data = json.load(f)

    filename_to_subset = {d["filename"]: d.get("subset", None) for d in ann_data}
elif dataset_name == "pope_test":
    ds = POPE_test_Dataset(
        data_dir=data_dir,
        annotation_file=annotation_file,
        split=split,
        dataset_size=dataset_size,
        seed=seed,
        dataset_name=dataset_name,
        mode="train",
        prompt_template="llava",
    )
else:
    raise ValueError(f"Unsupported dataset_name: {dataset_name}")

# -----------------------------
# MODEL WRAPPER
# -----------------------------
class Args:
    local_files_only = True
    cache_dir = None
    message_format = "role"

args = Args()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
set_seed(seed)

model_class = get_model_class(
    model_name_or_path=model_name_or_path,
    processor_name=processor_name,
    device=device,
    logger=None,
    args=args,
)

model = model_class.get_model()
tokenizer = model_class.get_tokenizer()

@torch.no_grad()
def generate_continuation(instruction, forced_response, continue_final_message, image_path):
    inputs = model_class.preprocessor(
        instruction=instruction,
        image_file=image_path,
        response=forced_response,
        generation_mode=True,
        continue_final_message=continue_final_message,
    )

    input_len = inputs["input_ids"].shape[1]
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,   # matches save_features.py generation path
    )

    continuation_ids = out[:, input_len:]
    continuation_text = tokenizer.batch_decode(continuation_ids, skip_special_tokens=True)[0]
    full_text = tokenizer.batch_decode(out, skip_special_tokens=True)[0]
    return continuation_text, full_text

rows = []
for i, item in enumerate(ds.data):  # remove [:3] for full dataset
    text = item["instruction"]
    gt_response = item["response"]

    # Positive (same answer)
    pos_instruction, pos_response, pos_continue = ds.construct_input(
        text=text,
        response=gt_response,
        force_answer=True,
        forced_answer_true=True,
        descriptive_answer=False,
        scenario=item.get("scenario", None),
    )

    # Negative (opposite answer)
    neg_instruction, neg_response, neg_continue = ds.construct_input(
        text=text,
        response=gt_response,
        force_answer=True,
        forced_answer_true=False,
        descriptive_answer=False,
        scenario=item.get("scenario", None),
    )

    pos_cont, pos_full = generate_continuation(
        pos_instruction, pos_response, pos_continue, item["image"]
    )
    neg_cont, neg_full = generate_continuation(
        neg_instruction, neg_response, neg_continue, item["image"]
    )

    def normalize_yes_no(s: str) -> str:
        s = s.strip().lower()
        if s.startswith("yes"):
            return "yes"
        if s.startswith("no"):
            return "no"
        raise ValueError(f"Cannot map to yes/no: {s}")

    pos_label = normalize_yes_no(pos_response)   # label implied by forced positive branch text
    neg_label = normalize_yes_no(neg_response)   # label implied by forced negative branch text
    gt_label  = normalize_yes_no(gt_response)    # dataset ground truth

    pos_text = join_prefix_and_cont(pos_response, pos_cont)
    neg_text = join_prefix_and_cont(neg_response, neg_cont)

    if pos_label == gt_label and neg_label != gt_label:
        not_hallucinated_text = pos_text
        hallucinated_text = neg_text
    elif neg_label == gt_label and pos_label != gt_label:
        not_hallucinated_text = neg_text
        hallucinated_text = pos_text
    else:
        raise ValueError(
            f"Ambiguous mapping: gt={gt_label}, pos_label={pos_label}, neg_label={neg_label}"
        )
    image_name = os.path.basename(item["image"])
    subset = filename_to_subset.get(image_name, None)

    rows.append({
        "idx": i,
        "image": image_name,
        "subset": subset,
        "scenario": item.get("scenario", None),
        "original_instruction": text,
        "ground_truth_response": gt_response,
        "forced_positive_response_text": pos_response,
        "forced_negative_response_text": neg_response,
        "positive_generated_continuation": pos_cont,
        "negative_generated_continuation": neg_cont,
        "positive_full_generated_text": pos_full,
        "negative_full_generated_text": neg_full,
        "hallucinated": hallucinated_text,
        "not_hallucinated": not_hallucinated_text,
    })

with open(output_json, "w") as f:
    json.dump(rows, f, indent=2)

print(f"Saved {len(rows)} rows to {output_json}")

src_root = /research/hal-afsharim/learn-to-steer/src
datasets exists: True
models exists: True


Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00,  4.41it/s]


Saved 6300 rows to pope_contrastive_with_continuations.json


In [13]:
import json
import re

with open("/research/hal-afsharim/learn-to-steer/src/examples/milad/pope_contrastive_with_continuations.json", "r") as f:
    data = json.load(f)

def extract_id_from_image(image_filename):
    match = re.search(r"(\d{12})", image_filename)
    return match.group(1) if match else image_filename.replace(".jpg", "").split("_")[-1]

with open("/research/hal-afsharim/learn-to-steer/src/examples/milad/pope_extracted.jsonl", "w") as f:
    for item in data:
        extracted = {
            "id": extract_id_from_image(item["image"]),
            "image": item["image"],
            "value": item["not_hallucinated"],
            "h_value": item["hallucinated"],
            "co_objects": [],
            "uncertain_objects": [],
            "question": item["original_instruction"].strip()
        }
        f.write(json.dumps(extracted, ensure_ascii=False) + "\n")

In [1]:
import os
import re
import json
import torch
import sys

src_root = os.path.abspath("../..")   # from src/examples/milad -> src
if src_root not in sys.path:
    sys.path.insert(0, src_root)

print("src_root =", src_root)
print("datasets exists:", os.path.exists(os.path.join(src_root, "datasets")))
print("models exists:", os.path.exists(os.path.join(src_root, "models")))

from datasets.image_text_dataset import POPE_train_Dataset, POPE_test_Dataset
from models import get_model_class
from helpers.utils import set_seed

# -----------------------------
# CONFIG
# -----------------------------
model_name_or_path = "llava-hf/llava-1.5-7b-hf"  # or "Qwen/Qwen2-VL-7B-Instruct"
processor_name = model_name_or_path

data_dir = "/research/hal-afsharim/learn-to-steer/data/pope/train"
annotation_file = "annotations.json"
dataset_name = "pope_train"                       # or "pope_test"
split = "all"                                     # train: all/adversarial/popular/random
dataset_size = -1
seed = 0

max_new_tokens = 100   # POPE extraction scripts use 100
output_json = "pope_continuation_prompts.json"

# optional: build subset map for either train or test annotations
ann_path = os.path.join(data_dir, annotation_file)
with open(ann_path, "r") as f:
    ann_data = json.load(f)
filename_to_subset = {d["filename"]: d.get("subset", None) for d in ann_data}

def normalize_yes_no(s: str) -> str:
    s = s.strip().lower()
    if s.startswith("yes"):
        return "yes"
    if s.startswith("no"):
        return "no"
    raise ValueError(f"Cannot map to yes/no: {s}")

def extract_yes_no_from_generation(text: str):
    """
    Parse model free-form output into yes/no.
    Returns 'yes'/'no' or None if unparsable.
    """
    t = text.strip().lower()
    # strongest signal: first token/word
    if t.startswith("yes"):
        return "yes"
    if t.startswith("no"):
        return "no"

    # fallback: first standalone yes/no anywhere
    m = re.search(r"\b(yes|no)\b", t)
    return m.group(1) if m else None



@torch.no_grad()
def generate_normal_response(instruction, image_path):
    # normal generation prompt: empty response, not continue_final_message
    inputs = model_class.preprocessor(
        instruction=instruction,
        image_file=image_path,
        response="",
        generation_mode=True,
        continue_final_message=False,
    )
    input_len = inputs["input_ids"].shape[1]

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,   # set False if you want deterministic output
    )

    generated_only = tokenizer.batch_decode(
        out[:, input_len:], skip_special_tokens=True
    )[0].strip()

    full_text = tokenizer.batch_decode(
        out, skip_special_tokens=True
    )[0].strip()

    return generated_only, full_text

# -----------------------------
# DATASET
# -----------------------------
if dataset_name == "pope_train":
    ds = POPE_train_Dataset(
        data_dir=data_dir,
        annotation_file=annotation_file,
        split=split,
        dataset_size=dataset_size,
        seed=seed,
        dataset_name=dataset_name,
        mode="train",
        prompt_template="llava",
    )
elif dataset_name == "pope_test":
    ds = POPE_test_Dataset(
        data_dir=data_dir,
        annotation_file=annotation_file,
        split=split,
        dataset_size=dataset_size,
        seed=seed,
        dataset_name=dataset_name,
        mode="train",
        prompt_template="llava",
    )
else:
    raise ValueError(f"Unsupported dataset_name: {dataset_name}")

# -----------------------------
# MODEL WRAPPER
# -----------------------------
class Args:
    local_files_only = True
    cache_dir = None
    message_format = "role"

args = Args()
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
set_seed(seed)

model_class = get_model_class(
    model_name_or_path=model_name_or_path,
    processor_name=processor_name,
    device=device,
    logger=None,
    args=args,
)

model = model_class.get_model()
tokenizer = model_class.get_tokenizer()

print("dataset size:", len(ds.data))

rows = []
for i, item in enumerate(ds.data):  # add [:N] for quick tests
    text = item["instruction"]
    gt_response = item["response"]
    gt_label = normalize_yes_no(gt_response)

    # normal (unforced) input construction
    instruction_, response_, continue_ = ds.construct_input(
        text=text,
        response=gt_response,
        force_answer=False,
        forced_answer_true=True,   # ignored when force_answer=False
        descriptive_answer=False,
        scenario=item.get("scenario", None),
    )

    # safety checks: should be empty response and False continue in normal mode
    # print(response_, continue_)  # expected: "", False

    gen_text, full_text = generate_normal_response(
        instruction=instruction_,
        image_path=item["image"],
    )

    pred_label = extract_yes_no_from_generation(gen_text)
    parse_ok = pred_label is not None

    is_hallucinated = None if not parse_ok else (pred_label != gt_label)
    hallucinated_text = gen_text if is_hallucinated is True else None
    not_hallucinated_text = gen_text if is_hallucinated is False else None

    image_name = os.path.basename(item["image"])
    subset = filename_to_subset.get(image_name, None)

    rows.append({
        "idx": i,
        "image": image_name,
        "subset": subset,
        "scenario": item.get("scenario", None),
        "original_instruction": text,
        "ground_truth_response": gt_response,
        "ground_truth_label": gt_label,
        "generated_response": gen_text,
        "full_generated_text": full_text,
        "predicted_label": pred_label,          # yes/no/None
        "parse_ok": parse_ok,
        "is_hallucinated": is_hallucinated,     # True/False/None
        "hallucinated": hallucinated_text,
        "not_hallucinated": not_hallucinated_text,
    })

with open(output_json, "w") as f:
    json.dump(rows, f, indent=2)

print(f"Saved {len(rows)} rows to {output_json}")

src_root = /research/hal-afsharim/learn-to-steer/src
datasets exists: True
models exists: True


/research/hal-afsharim/miniconda3/envs/xl_vlm/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/research/hal-afsharim/miniconda3/envs/xl_vlm/lib/python3.9/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
/research/hal-afsharim/miniconda3/envs/xl_vlm/lib/python3.9/site-packages/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging
Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00,  4.84it/s]


dataset size: 6300


Expanding inputs for image tokens in LLaVa should be done in processing. Please add `patch_size` and `vision_feature_select_strategy` to the model's processing config or set directly with `processor.patch_size = {{patch_size}}` and processor.vision_feature_select_strategy = {{vision_feature_select_strategy}}`. Using processors without these attributes in the config is deprecated and will throw an error in v4.50.


Saved 6300 rows to pope_continuation_prompts.json


In [2]:
import json
import re

# use your normal-generation output file here
with open("/research/hal-afsharim/learn-to-steer/src/examples/milad/pope_continuation_prompts.json", "r") as f:
    data = json.load(f)

def extract_id_from_image(image_filename):
    match = re.search(r"(\d{12})", image_filename)
    return match.group(1) if match else image_filename.replace(".jpg", "").split("_")[-1]

num_total = len(data)
num_written = 0

with open("/research/hal-afsharim/learn-to-steer/src/examples/milad/pope_extracted_prompt_6300.jsonl", "w") as f: 
    for item in data:
        if not item.get("parse_ok", False):
            continue

        extracted = {
            "id": extract_id_from_image(item["image"]),
            "image": item["image"],
            "value": item["not_hallucinated"],
            "h_value": item["hallucinated"],
            "co_objects": [],
            "uncertain_objects": [],
            "question": item["original_instruction"].strip()
        }
        f.write(json.dumps(extracted, ensure_ascii=False) + "\n")
        num_written += 1

print(f"Wrote {num_written}/{num_total} rows with parse_ok=true")

Wrote 6260/6300 rows with parse_ok=true


In [3]:
import json
import re

# use your normal-generation output file here
with open("/research/hal-afsharim/learn-to-steer/src/examples/milad/pope_continuation_prompts_1000.json", "r") as f:
    data = json.load(f)

def extract_id_from_image(image_filename):
    match = re.search(r"(\d{12})", image_filename)
    return match.group(1) if match else image_filename.replace(".jpg", "").split("_")[-1]

num_total = len(data)
num_written = 0

with open("/research/hal-afsharim/learn-to-steer/src/examples/milad/pope_extracted_prompt_1000.jsonl", "w") as f: 
    for item in data:
        if not item.get("parse_ok", False):
            continue

        extracted = {
            "id": extract_id_from_image(item["image"]),
            "image": item["image"],
            "value": item["not_hallucinated"],
            "h_value": item["hallucinated"],
            "co_objects": [],
            "uncertain_objects": [],
            "question": item["original_instruction"].strip()
        }
        f.write(json.dumps(extracted, ensure_ascii=False) + "\n")
        num_written += 1

print(f"Wrote {num_written}/{num_total} rows with parse_ok=true")

Wrote 996/1000 rows with parse_ok=true


In [8]:
import json

path = "/research/hal-afsharim/learn-to-steer/src/examples/milad/pope_extracted_prompt_1000.jsonl"

n_total = 0
n_hall = 0
n_nonhall = 0
n_other = 0

with open(path, "r") as f:
    for line in f:
        if not line.strip():
            continue
        item = json.loads(line)
        n_total += 1

        # Your format: value = non-hall text or null, h_value = hall text or null
        has_nonhall = item.get("value") is not None
        has_hall = item.get("h_value") is not None

        if has_hall:
            n_hall += 1
        elif has_nonhall:
            n_nonhall += 1

if n_total > 0:
    print(f"Hallucinated %: {100*n_hall/n_total:.2f}%")
    print(f"Non-hallucinated %: {100*n_nonhall/n_total:.2f}%")
    print(f"Number of hallucinated: {n_hall}")
    print(f"Number of non-hallucinated: {n_nonhall}")
    print(f"Number of total: {n_total}")

Hallucinated %: 20.98%
Non-hallucinated %: 79.02%
Number of hallucinated: 209
Number of non-hallucinated: 787
Number of total: 996
